# Agent Memory with Neo4j

Language models do not remember anything between conversations. This notebook shows how to add memory using the Neo4j Agent Memory Python SDK.

You will build it up in three steps.

| Part | What you add | What the agent gains |
|------|--------------|----------------------|
| 1 | Short-term memory | It can follow the current conversation |
| 2 | Entity extraction | Sentences become a graph of people, places and things |
| 3 | Preferences | It remembers what a user likes, across sessions |

After each part there are Cypher queries so you can see what landed in the database.

Everything runs against your own Neo4j instance. See the README for setup.

## Setup

Install the SDK.

In [ ]:
%pip install --quiet "neo4j-agent-memory[gliner, openai]" python-dotenv

#optional if you want spacy extractor
# pip install spacy && python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Connect to Neo4j.

The SDK is asynchronous, so every call needs `await`. In a script you would wrap this in `asyncio.run()`. Jupyter already runs an event loop, so `await` works directly in a cell.

We open the connection once here and close it in the last cell.

In [1]:
import os
import warnings

from dotenv import load_dotenv
from neo4j_agent_memory import MemoryClient, MemorySettings

load_dotenv()
warnings.filterwarnings("ignore")

NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

settings = MemorySettings(
    neo4j={
        "uri": NEO4J_URI,
        "username": NEO4J_USER,
        "password": NEO4J_PASSWORD,
        "database": NEO4J_DATABASE,
    },
)

client = MemoryClient(settings)
await client.__aenter__()

print(f"Connected to {NEO4J_URI}")

Connected to neo4j+s://67b905b6.databases.neo4j.io


A small helper so we can run Cypher queries and see the results as a table. This is not part of the SDK, it is just a way to look inside the database.

In [2]:
from neo4j import GraphDatabase

_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def cypher(query, **params):
    """Run a Cypher query and print the rows as a table."""
    with _driver.session(database=NEO4J_DATABASE) as session:
        rows = [record.data() for record in session.run(query, **params)]

    if not rows:
        print("(no rows)")
        return

    columns = list(rows[0].keys())
    widths = {
        c: min(60, max(len(str(c)), max(len(str(r.get(c, ""))) for r in rows)))
        for c in columns
    }

    header = "  ".join(str(c).ljust(widths[c]) for c in columns)
    print(header)
    print("-" * len(header))
    for row in rows:
        print("  ".join(str(row.get(c, ""))[:60].ljust(widths[c]) for c in columns))
    print(f"\n{len(rows)} row(s)")

cypher("RETURN 'Neo4j is reachable' AS status")

status            
------------------
Neo4j is reachable

1 row(s)


## Part 1 - Short-term memory

Short-term memory is the conversation itself: messages, in order, grouped into a session.

Think of `session_id` as a folder name. Everything stored under the same `session_id` belongs to the same conversation. A different user, or the same user returning tomorrow, gets a different one.

Storing a message takes one call.

In [3]:
SESSION_1 = "workshop-session-001"

message = await client.short_term.add_message(
    session_id=SESSION_1,
    role="user",
    content="Hello! I'm looking for running shoes. I really like Nike products.",
)

print("Stored message")
print("  id:      ", message.id)
print("  session: ", SESSION_1)

Stage 'SpacyEntityExtractor' failed: spaCy is required for SpacyEntityExtractor. Install with: pip install spacy && python -m spacy download en_core_web_sm
2026-09-02 10:09:06.528599601 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
Fetching 12 files: 100%|██████████| 12/12 [00:42<00:00,  3.57s/it]


Stored message
  id:       920c0ada-ce12-42d9-b40f-4f6faa0e05f7
  session:  workshop-session-001


Read the conversation back. This is what a chatbot does on every turn, so the model has the thread so far.

In [4]:
conversation = await client.short_term.get_conversation(session_id=SESSION_1)

print(f"{len(conversation.messages)} message(s) in this conversation\n")

for msg in conversation.messages:
    print(f"[{msg.role}] {msg.content}")

1 message(s) in this conversation

[MessageRole.USER] Hello! I'm looking for running shoes. I really like Nike products.


### Look at the graph

So far this behaves like any chat history store. The difference is where it lives.

In [5]:
cypher("""
MATCH (m:Message)
RETURN m.role AS role, m.content AS content
ORDER BY m.created_at
""")

role  content                                                     
------------------------------------------------------------------
user  Hello! I'm looking for running shoes. I really like Nike pro

1 row(s)


In [6]:
# Which node types exist so far, and how many of each
cypher("""
MATCH (n)
RETURN labels(n) AS labels, count(*) AS count
ORDER BY count DESC
""")

labels                                 count
--------------------------------------------
['Conversation']                       1    
['Message']                            1    
['Entity', 'Object']                   1    
['Organization', 'Company', 'Entity']  1    

4 row(s)


You should see `Conversation` and `Message` nodes. The messages are chained together, so the order of the conversation is held in the graph itself.

## Part 2 - Entity extraction

This is where it stops being a chat log.

A message is just text. `"I've been using Nike Air Max shoes"` means nothing to a database on its own. Entity extraction reads that sentence, picks out the things in it, and stores them as nodes you can query and connect.

### Choosing an extractor

The SDK offers several, at different speeds and costs.

| Extractor | Speed | Cost | Good for |
|-----------|-------|------|----------|
| spaCy | Very fast | Free | Common types: people, organizations, places |
| GLiNER | Fast | Free | Types you define yourself. A good default. |
| GLiREL | Fast | Free | Finding relationships between entities |
| LLM | Slow | Paid | Text the others cannot handle |

We use GLiNER. It runs locally, costs nothing per message, and finds whatever entity types you ask for without retraining.

`for_poleo()` sets it up with the POLE+O types: Person, Object, Location, Event and Organization. Most things people mention in conversation fit one of those five.

### Run the extraction

Entity extraction runs as a separate script. Open a terminal in this folder and run:

```bash
python entity_extraction.py
```

The first run downloads the GLiNER model, which takes a minute or two. It is cached after that.

The script stores four messages and the entities found in them, then prints a summary. Once it finishes, come back here and run the cells below to see the result in the graph.

### Look at the graph

In [7]:
# Entities grouped by type
cypher("""
MATCH (e:Entity)
RETURN e.type AS type, collect(e.name) AS entities, count(*) AS count
ORDER BY count DESC
""")

type          entities           count
--------------------------------------
OBJECT        ['running shoes']  1    
ORGANIZATION  ['Nike']           1    

2 row(s)


In [8]:
# Each entity carries several labels.
# An organization is not just :Entity, it is :Entity:Organization:Company.
# That is what lets you query at whichever level of precision you need.
cypher("""
MATCH (e:Entity)
RETURN e.name AS name, labels(e) AS labels
LIMIT 15
""")

name           labels                               
----------------------------------------------------
running shoes  ['Entity', 'Object']                 
Nike           ['Organization', 'Company', 'Entity']

2 row(s)


In [9]:
# Messages linked to the entities mentioned in them.
# This returns rows only if entities were linked back to their source message.
cypher("""
MATCH (m:Message)-[:MENTIONS]->(e:Entity)
RETURN m.content AS message, e.name AS entity, e.type AS type
LIMIT 20
""")

message                                                       entity         type        
-----------------------------------------------------------------------------------------
Hello! I'm looking for running shoes. I really like Nike pro  running shoes  OBJECT      
Hello! I'm looking for running shoes. I really like Nike pro  Nike           ORGANIZATION

2 row(s)


## Part 3 - Preferences

Entities are facts about the world. Preferences are facts about the user: what they like, what they can spend, how they want to be treated.

These are the things that should outlive a session. If someone told you last week that they only wear Nike and their budget is $150, asking again today is a poor experience.

Preferences are stored as `category` and `preference` pairs.

In [10]:
preferences = [
    ("brand",    "Prefers Nike products"),
    ("budget",   "Budget around $150"),
    ("style",    "Prefers lightweight shoes"),
    ("activity", "Trains for marathons"),
    ("location", "Lives in San Francisco"),
]

for category, preference in preferences:
    await client.long_term.add_preference(category=category, preference=preference)
    print(f"stored  [{category}] {preference}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"


stored  [brand] Prefers Nike products


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replac

stored  [budget] Budget around $150
stored  [style] Prefers lightweight shoes


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"


stored  [activity] Trains for marathons


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"


stored  [location] Lives in San Francisco


In [11]:
# Confirm they are in the graph
cypher("""
MATCH (p:Preference)
RETURN p.category AS category, p.preference AS preference
ORDER BY p.category
""")

category  preference               
-----------------------------------
activity  Trains for marathons     
brand     Prefers Nike products    
budget    Budget around $150       
location  Lives in San Francisco   
style     Prefers lightweight shoes

5 row(s)


### Building context for the model

This is where it comes together, and it is smaller than people expect.

When a user asks something, you search memory for what is relevant, format it as text, and put it in front of the model. That text is what turns a generic assistant into one that knows this particular person.

Note that the user's question is used as the search query. We are not sending everything we know, only what is relevant to this question.

In [14]:
async def build_context(question: str) -> str:
    """Fetch the memory relevant to a question and format it for a prompt."""
    prefs = await client.long_term.search_preferences(query=question)

    lines = ["## What we know about this user", ""]
    for pref in prefs:
        lines.append(f"- {pref.category}: {pref.preference}")

    return "\n".join(lines)


question = "Can you recommend some running shoes?"
context = await build_context(question)

print(context)
print()
print("=" * 60)
print("This block goes into the system prompt, above the user's question:")
print(f'  "{question}"')

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\nRETURN node AS p, score\nORDER BY score DESC\n"


## What we know about this user

- style: Prefers lightweight shoes
- brand: Prefers Nike products

This block goes into the system prompt, above the user's question:
  "Can you recommend some running shoes?"


Compare two answers to that question.

Without memory, the model returns a generic list of running shoes.

With the block above in front of it, it can suggest lightweight Nike shoes around $150, suitable for marathon training, Same model, same question. The difference is the context.

In [15]:
# Everything built so far, counted by node type
cypher("""
MATCH (n)
RETURN labels(n) AS labels, count(*) AS count
ORDER BY count DESC
""")

labels                                 count
--------------------------------------------
['Preference']                         5    
['Conversation']                       2    
['Message']                            2    
['Entity', 'Object']                   1    
['Organization', 'Company', 'Entity']  1    

5 row(s)


## Conclusion

In three steps you built a memory layer an agent could use.

Short-term memory gave you the conversation: messages in order, grouped by session, so the agent can follow a thread.

Entity extraction turned those sentences into a graph. GLiNER found the people, brands, places and products, typed them with POLE+O, and stored them as connected nodes. It ran locally, with no cost per message.

Preferences gave the agent something to remember about the person rather than the conversation. That is the part that survives a session ending.

`build_context()` showed the payoff. Retrieving memory and formatting it for a prompt is only a few lines. The value comes from having stored things in a structure worth retrieving from.

### Not covered here

There is a third memory type, reasoning memory, which records the steps and tool calls an agent made and whether they succeeded. It is what allows an agent to improve rather than only recall.

Everything here ran against a local Neo4j. The hosted service, NAMS, offers the same API with extraction and deduplication handled server-side, so you only need an API key. Moving between the two is one environment variable.

We also skipped deduplication, which keeps "Nike", "Nike Inc" and "nike" from becoming three separate nodes. The README links to the guide.

## Cleanup

Close the connections when you are done.

In [16]:
await client.__aexit__(None, None, None)
_driver.close()
print("Connections closed")

Connections closed
